## Dataset Loading and preparation

In [35]:
# Section 1: Imports & Paths
import pandas as pd
import numpy as np
import os
import pickle
import json
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import KFold, train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score, roc_curve
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.ensemble import RandomForestClassifier
import shap
import optuna
import warnings
warnings.filterwarnings("ignore")

# Paths
DATA_PATH = "E:\Thesis\llm-ids-shield\Data\Processed\Preprocessed_file.csv"   # Change to your dataset path
RESULTS_DIR = r"E:\Thesis\llm-ids-shield\Notebooks\results"
SHAP_DIR = os.path.join(RESULTS_DIR, "shap_plots")
os.makedirs(SHAP_DIR, exist_ok=True)


In [36]:
# Section 2: Load Dataset
df = pd.read_csv(DATA_PATH)

# Encode labels if they are categorical
if df['Label'].dtype == 'object':
    le = LabelEncoder()
    df['Label'] = le.fit_transform(df['Label'])

# Separate features and target
X = df.drop('Label', axis=1)
y = df['Label']

print("Dataset Shape:", df.shape)
print("Feature Columns:", X.columns.tolist())


Dataset Shape: (625783, 23)
Feature Columns: ['Flow_Duration', 'Init_Bwd_Win_Byts', 'Dst_Port', 'Idle_Mean', 'Flow_IAT_Min', 'Flow_Pkts/s', 'Pkt_Len_Max', 'ACK_Flag_Cnt', 'Fwd_Pkt_Len_Max', 'Bwd_Header_Len', 'TotLen_Bwd_Pkts', 'Fwd_Pkts/s', 'Flow_Byts/s', 'Bwd_Pkt_Len_Max', 'Bwd_Pkts/s', 'TotLen_Fwd_Pkts', 'Flow_IAT_Max', 'Fwd_Pkt_Len_Min', 'Bwd_Pkt_Len_Mean', 'Pkt_Len_Std', 'SYN_Flag_Cnt', 'Pkt_Len_Mean']


In [37]:
# Section 3: Utility Functions

# 1. Metrics calculation
def compute_metrics(y_true, y_pred, y_prob=None, classes=None):
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average='weighted')
    rec = recall_score(y_true, y_pred, average='weighted')
    f1 = f1_score(y_true, y_pred, average='weighted')
    cm = confusion_matrix(y_true, y_pred)

    # Only compute ROC AUC if y_prob is valid
    roc_auc = None
    if y_prob is not None:
        try:
            roc_auc = roc_auc_score(y_true, y_prob, multi_class='ovr')
        except ValueError:
            print("Skipping ROC AUC for this model/fold due to class mismatch.")
    
    return {"accuracy": acc, "precision": prec, "recall": rec, "f1": f1, "confusion_matrix": cm, "roc_auc": roc_auc}


# 2. Plot Confusion Matrix
def plot_confusion_matrix(cm, labels, title="Confusion Matrix"):
    plt.figure(figsize=(8,6))
    sns.heatmap(cm, annot=True, fmt="d", xticklabels=labels, yticklabels=labels, cmap="Blues")
    plt.title(title)
    plt.ylabel('True')
    plt.xlabel('Predicted')
    plt.show()

# 3. Save model
def save_model(model, filename):
    with open(filename, "wb") as f:
        pickle.dump(model, f)

# 4. SHAP Explainability
def shap_explain(model, X, model_name="model"):
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X)
    shap.summary_plot(shap_values, X, show=False)
    plt.savefig(os.path.join(SHAP_DIR, f"{model_name}_shap_summary.png"))
    plt.close()


In [ ]:
# Section 4: Optuna Tuning
def optuna_tuning_xgb(X, y, n_trials=30):
    def objective(trial):
        param = {
            "n_estimators": trial.suggest_int("n_estimators", 100, 500),
            "max_depth": trial.suggest_int("max_depth", 3, 15),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3),
            "subsample": trial.suggest_float("subsample", 0.5, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
            "gamma": trial.suggest_float("gamma", 0, 5)
        }
        kf = KFold(n_splits=3, shuffle=True, random_state=42)
        f1_scores = []
        for train_idx, val_idx in kf.split(X):
            X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
            y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
            model = XGBClassifier(**param, use_label_encoder=False, eval_metric='mlogloss')
            model.fit(X_train, y_train)
            y_pred = model.predict(X_val)
            f1_scores.append(f1_score(y_val, y_pred, average='weighted'))
        return np.mean(f1_scores)
    
    study = optuna.create_study(direction='maximize')
    study.optimize(objective, n_trials=n_trials)
    return study.best_params

def optuna_tuning_lgb(X, y, n_trials=30):
    def objective(trial):
        param = {
            "n_estimators": trial.suggest_int("n_estimators", 100, 500),
            "max_depth": trial.suggest_int("max_depth", 3, 15),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3),
            "num_leaves": trial.suggest_int("num_leaves", 20, 150),
            "subsample": trial.suggest_float("subsample", 0.5, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0)
        }
        kf = KFold(n_splits=3, shuffle=True, random_state=42)
        f1_scores = []
        for train_idx, val_idx in kf.split(X):
            X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
            y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
            model = LGBMClassifier(**param)
            model.fit(X_train, y_train)
            y_pred = model.predict(X_val)
            f1_scores.append(f1_score(y_val, y_pred, average='weighted'))
        return np.mean(f1_scores)
    
    study = optuna.create_study(direction='maximize')
    study.optimize(objective, n_trials=n_trials)
    return study.best_params



'def optuna_tuning_rf(X, y, n_trials=30):\n    def objective(trial):\n        param = {\n            "n_estimators": trial.suggest_int("n_estimators", 100, 500),\n            "max_depth": trial.suggest_int("max_depth", 3, 30),\n            "min_samples_split": trial.suggest_int("min_samples_split", 2, 10),\n            "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 5)\n        }\n        kf = KFold(n_splits=3, shuffle=True, random_state=42)\n        f1_scores = []\n        for train_idx, val_idx in kf.split(X):\n            X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]\n            y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]\n            model = RandomForestClassifier(**param)\n            model.fit(X_train, y_train)\n            y_pred = model.predict(X_val)\n            f1_scores.append(f1_score(y_val, y_pred, average=\'weighted\'))\n        return np.mean(f1_scores)\n\n    study = optuna.create_study(direction=\'maximize\')\n    study.optimize(objective, 

In [51]:
def optuna_tuning_rf(X, y, n_trials=30):
    def objective(trial):
        param = {
            "n_estimators": trial.suggest_int("n_estimators", 100, 500),
            "max_depth": trial.suggest_int("max_depth", 3, 30),
            "min_samples_split": trial.suggest_int("min_samples_split", 2, 10),
            "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 5)
        }
        kf = KFold(n_splits=3, shuffle=True, random_state=42)
        f1_scores = []
        for train_idx, val_idx in kf.split(X):
            X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
            y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
            model = RandomForestClassifier(**param,n_jobs=-1,)
            model.fit(X_train, y_train)
            y_pred = model.predict(X_val)
            f1_scores.append(f1_score(y_val, y_pred, average='weighted'))
        return np.mean(f1_scores)
    
    study = optuna.create_study(direction='maximize')
    study.optimize(objective, n_trials=n_trials)
    return study.best_params


In [39]:
# Section 5: k-Fold Training
def train_models_kfold(X, y, models_dict, k=5):
    kf = KFold(n_splits=k, shuffle=True, random_state=42)
    all_metrics = {name: [] for name in models_dict.keys()}
    fold_num = 1

    for train_idx, val_idx in kf.split(X):
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

        for name, model in models_dict.items():
            model.fit(X_train, y_train)
            y_pred = model.predict(X_val)
            y_prob = model.predict_proba(X_val) if hasattr(model, "predict_proba") else None
            metrics = compute_metrics(y_val, y_pred, y_prob)
            metrics['fold'] = fold_num
            all_metrics[name].append(metrics)
        fold_num += 1
    return all_metrics


In [40]:
# json serializable conversion
import numpy as np
import json

def make_json_serializable(metrics_all):
    serializable = {}
    for model, folds in metrics_all.items():
        serializable[model] = []
        for fold in folds:
            fold_copy = {}
            for k, v in fold.items():
                if isinstance(v, np.ndarray):
                    fold_copy[k] = v.tolist()  # convert array to list
                elif isinstance(v, (np.float64, np.float32)):
                    fold_copy[k] = float(v)    # convert to float
                elif isinstance(v, (np.int64, np.int32)):
                    fold_copy[k] = int(v)      # convert to int
                else:
                    fold_copy[k] = v
            serializable[model].append(fold_copy)
    return serializable



# Model Training

### RandomForest initial run with k-fold

In [52]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(n_jobs=-1, random_state=42)
rf_metrics = train_models_kfold(X, y, {"RandomForest": rf_model}, k=5)

# Convert metrics to JSON-serializable format
rf_metrics_serializable = make_json_serializable(rf_metrics)

# Save metrics
with open(os.path.join(RESULTS_DIR, "rf_metrics.json"), "w") as f:
    json.dump(rf_metrics_serializable, f, indent=4)

# Save the trained RandomForest model
# save_model(rf_model, os.path.join(RESULTS_DIR, "rf_initial_model.pkl"))

print("RandomForest initial run done. Metrics saved in rf_metrics.json")


RandomForest initial run done. Metrics saved in rf_metrics.json


### XGBoost initial run with k-fold

In [42]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(use_label_encoder=False, eval_metric='mlogloss', random_state=42)
xgb_metrics = train_models_kfold(X, y, {"XGBoost": xgb_model}, k=5)

# Convert metrics to JSON-serializable format
xgb_metrics = make_json_serializable(xgb_metrics)
with open(os.path.join(RESULTS_DIR, "xgb_metrics.json"), "w") as f:
    json.dump(xgb_metrics, f, indent=4)

# Save the trained XGBoost model
# save_model(xgb_model, os.path.join(RESULTS_DIR, "xgb_initial_model.pkl"))
print("XGBoost initial run done. Metrics saved in xgb_metrics.json")


XGBoost initial run done. Metrics saved in xgb_metrics.json


### Lightgbm initial run with k-fold

In [43]:
from lightgbm import LGBMClassifier

lgb_model = LGBMClassifier(random_state=42)
lgb_metrics = train_models_kfold(X, y, {"LightGBM": lgb_model}, k=5)

# Convert metrics to JSON-serializable format
lgb_metrics = make_json_serializable(lgb_metrics)
with open(os.path.join(RESULTS_DIR, "lgb_metrics.json"), "w") as f:
    json.dump(lgb_metrics, f, indent=4)


print("LightGBM initial run done. Metrics saved in lgb_metrics.json")


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.036026 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4723
[LightGBM] [Info] Number of data points in the train set: 500626, number of used features: 22
[LightGBM] [Info] Start training from score -2.359095
[LightGBM] [Info] Start training from score -2.869118
[LightGBM] [Info] Start training from score -2.425439
[LightGBM] [Info] Start training from score -2.418282
[LightGBM] [Info] Start training from score -1.642438
[LightGBM] [Info] Start training from score -1.225563
[LightGBM] [Info] Start training from score -2.748250
[LightGBM] [Info] Start training from score -3.340263
[LightGBM] [Info] Start training from score -2.469065
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.008607 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`

In [48]:

# XGBoost hyperparameter tuning
best_xgb_params = optuna_tuning_xgb(X, y, n_trials=10)
print("Best XGB params:", best_xgb_params)

xgb_best = XGBClassifier(**best_xgb_params, use_label_encoder=False, eval_metric='mlogloss', n_jobs=-1, random_state=42)
xgb_best_metrics = train_models_kfold(X, y, {"XGBoost": xgb_best}, k=3)
xgb_best_metrics = make_json_serializable(xgb_best_metrics)
save_model(xgb_best, os.path.join(RESULTS_DIR, "xgb_best_model.pkl"))
with open(os.path.join(RESULTS_DIR, "xgb_best_params.json"), "w") as f:
    json.dump(best_xgb_params, f, indent=4)
with open(os.path.join(RESULTS_DIR, "xgb_best_metrics.json"), "w") as f:
    json.dump(xgb_best_metrics, f, indent=4)

print("XGBoost trained with best params. Model & metrics saved.")

[I 2025-11-23 15:51:48,705] A new study created in memory with name: no-name-b16e4fd5-5321-4efa-9d06-fe260022a796
[I 2025-11-23 15:53:45,493] Trial 0 finished with value: 0.6640953875608056 and parameters: {'n_estimators': 219, 'max_depth': 3, 'learning_rate': 0.25262260565154476, 'subsample': 0.8829706714328482, 'colsample_bytree': 0.7027112716607036, 'gamma': 2.731879002301245}. Best is trial 0 with value: 0.6640953875608056.
[I 2025-11-23 16:06:39,911] Trial 1 finished with value: 0.6390508107319658 and parameters: {'n_estimators': 436, 'max_depth': 13, 'learning_rate': 0.2753518834194989, 'subsample': 0.9099511347250004, 'colsample_bytree': 0.98843942795382, 'gamma': 0.019133872419547227}. Best is trial 0 with value: 0.6640953875608056.
[I 2025-11-23 16:08:21,275] Trial 2 finished with value: 0.6498552057410566 and parameters: {'n_estimators': 124, 'max_depth': 9, 'learning_rate': 0.295464245077693, 'subsample': 0.8680460044890932, 'colsample_bytree': 0.9611527746768552, 'gamma': 0

Best XGB params: {'n_estimators': 299, 'max_depth': 4, 'learning_rate': 0.09704895045820416, 'subsample': 0.6049699729187152, 'colsample_bytree': 0.9291454831381345, 'gamma': 1.8827187166209343}
XGBoost trained with best params. Model & metrics saved.


In [53]:
# RandomForest hyperparameter tuning
best_rf_params = optuna_tuning_rf(X, y, n_trials=20)
print("Best RF params:", best_rf_params)
rf_best = RandomForestClassifier(**best_rf_params, n_jobs=-1, random_state=42)
rf_best_metrics = train_models_kfold(X, y, {"RandomForest": rf_best}, k=5)
rf_best_metrics = make_json_serializable(rf_best_metrics)
save_model(rf_best, os.path.join(RESULTS_DIR, "rf_best_model.pkl"))
with open(os.path.join(RESULTS_DIR, "rf_best_params.json"), "w") as f:
    json.dump(best_rf_params, f, indent=4)
with open(os.path.join(RESULTS_DIR, "rf_best_metrics.json"), "w") as f:
    json.dump(rf_best_metrics, f, indent=4)
print("RandomForest trained with best params. Model & metrics saved.")


[I 2025-11-23 17:55:44,057] A new study created in memory with name: no-name-d1891cfa-d120-4617-9834-0482b2c1f9de
[I 2025-11-23 18:00:06,206] Trial 0 finished with value: 0.6351969070106335 and parameters: {'n_estimators': 423, 'max_depth': 15, 'min_samples_split': 6, 'min_samples_leaf': 5}. Best is trial 0 with value: 0.6351969070106335.
[I 2025-11-23 18:01:07,855] Trial 1 finished with value: 0.6356152363160916 and parameters: {'n_estimators': 104, 'max_depth': 13, 'min_samples_split': 4, 'min_samples_leaf': 4}. Best is trial 1 with value: 0.6356152363160916.
[I 2025-11-23 18:06:34,341] Trial 2 finished with value: 0.6423043476751582 and parameters: {'n_estimators': 447, 'max_depth': 24, 'min_samples_split': 10, 'min_samples_leaf': 2}. Best is trial 2 with value: 0.6423043476751582.
[I 2025-11-23 18:08:20,667] Trial 3 finished with value: 0.6430880476872541 and parameters: {'n_estimators': 144, 'max_depth': 28, 'min_samples_split': 10, 'min_samples_leaf': 5}. Best is trial 3 with val

Best RF params: {'n_estimators': 267, 'max_depth': 22, 'min_samples_split': 3, 'min_samples_leaf': 5}
RandomForest trained with best params. Model & metrics saved.


## LightGBM hyperparameter tuning

In [49]:

best_lgb_params = optuna_tuning_lgb(X, y, n_trials=10)
print("Best LGB params:", best_lgb_params)


[I 2025-11-23 16:39:32,319] A new study created in memory with name: no-name-54eb9395-a1b3-4322-a9b6-1100e07f9767


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.053495 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4720
[LightGBM] [Info] Number of data points in the train set: 417188, number of used features: 22
[LightGBM] [Info] Start training from score -2.360203
[LightGBM] [Info] Start training from score -2.864230
[LightGBM] [Info] Start training from score -2.428991
[LightGBM] [Info] Start training from score -2.417578
[LightGBM] [Info] Start training from score -1.641350
[LightGBM] [Info] Start training from score -1.225377
[LightGBM] [Info] Start training from score -2.747601
[LightGBM] [Info] Start training from score -3.340939
[LightGBM] [Info] Start training from score -2.471497
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf

[I 2025-11-23 16:41:26,547] Trial 0 finished with value: 0.6424469845217087 and parameters: {'n_estimators': 211, 'max_depth': 3, 'learning_rate': 0.016450621945713804, 'num_leaves': 107, 'subsample': 0.662756416811145, 'colsample_bytree': 0.5519121519201315}. Best is trial 0 with value: 0.6424469845217087.


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.032446 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4720
[LightGBM] [Info] Number of data points in the train set: 417188, number of used features: 22
[LightGBM] [Info] Start training from score -2.360203
[LightGBM] [Info] Start training from score -2.864230
[LightGBM] [Info] Start training from score -2.428991
[LightGBM] [Info] Start training from score -2.417578
[LightGBM] [Info] Start training from score -1.641350
[LightGBM] [Info] Start training from score -1.225377
[LightGBM] [Info] Start training from score -2.747601
[LightGBM] [Info] Start training from score -3.340939
[LightGBM] [Info] Start training from score -2.471497
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf

[I 2025-11-23 16:44:17,057] Trial 1 finished with value: 0.43417714303482935 and parameters: {'n_estimators': 247, 'max_depth': 13, 'learning_rate': 0.22841596296703065, 'num_leaves': 32, 'subsample': 0.5478234555086028, 'colsample_bytree': 0.8295709893792289}. Best is trial 0 with value: 0.6424469845217087.


...

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.032838 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4720
[LightGBM] [Info] Number of data points in the train set: 417188, number of used features: 22
[LightGBM] [Info] Start training from score -2.360203
[LightGBM] [Info] Start training from score -2.864230
[LightGBM] [Info] Start training from score -2.428991
[LightGBM] [Info] Start training from score -2.417578
[LightGBM] [Info] Start training from score -1.641350
[LightGBM] [Info] Start training from score -1.225377
[LightGBM] [Info] Start training from score -2.747601
[LightGBM] [Info] Start training from score -3.340939
[LightGBM] [Info] Start training from score -2.471497
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf

[I 2025-11-23 17:04:42,436] Trial 8 finished with value: 0.6521422997340972 and parameters: {'n_estimators': 249, 'max_depth': 14, 'learning_rate': 0.08943651125639032, 'num_leaves': 93, 'subsample': 0.6019245180386366, 'colsample_bytree': 0.8566172362661808}. Best is trial 3 with value: 0.6703063848061731.


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.036229 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4720
[LightGBM] [Info] Number of data points in the train set: 417188, number of used features: 22
[LightGBM] [Info] Start training from score -2.360203
[LightGBM] [Info] Start training from score -2.864230
[LightGBM] [Info] Start training from score -2.428991
[LightGBM] [Info] Start training from score -2.417578
[LightGBM] [Info] Start training from score -1.641350
[LightGBM] [Info] Start training from score -1.225377
[LightGBM] [Info] Start training from score -2.747601
[LightGBM] [Info] Start training from score -3.340939
[LightGBM] [Info] Start training from score -2.471497
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf

[I 2025-11-23 17:06:11,561] Trial 9 finished with value: 0.6652487982728711 and parameters: {'n_estimators': 164, 'max_depth': 3, 'learning_rate': 0.1550472158009875, 'num_leaves': 109, 'subsample': 0.6601696206182281, 'colsample_bytree': 0.9477842773111373}. Best is trial 3 with value: 0.6703063848061731.


Best LGB params: {'n_estimators': 127, 'max_depth': 6, 'learning_rate': 0.04513113781346488, 'num_leaves': 90, 'subsample': 0.679018960028567, 'colsample_bytree': 0.692802143394516}


In [50]:
# Build final LightGBM model
lgb_best = LGBMClassifier(**best_lgb_params, n_jobs=-1, random_state=42)

# Train with K-Fold
lgb_best_metrics = train_models_kfold(X, y, {"LightGBM": lgb_best}, k=3)

# Convert metrics to JSON-safe format
lgb_best_metrics = make_json_serializable(lgb_best_metrics)

# Save model
save_model(lgb_best, os.path.join(RESULTS_DIR, "lgb_best_model.pkl"))

# Save best params
with open(os.path.join(RESULTS_DIR, "lgb_best_params.json"), "w") as f:
    json.dump(best_lgb_params, f, indent=4)

# Save metrics
with open(os.path.join(RESULTS_DIR, "lgb_best_metrics.json"), "w") as f:
    json.dump(lgb_best_metrics, f, indent=4)

print("LightGBM trained with best params. Model & metrics saved.")


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.029997 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4722
[LightGBM] [Info] Number of data points in the train set: 417188, number of used features: 22
[LightGBM] [Info] Start training from score -2.360203
[LightGBM] [Info] Start training from score -2.864230
[LightGBM] [Info] Start training from score -2.428991
[LightGBM] [Info] Start training from score -2.417578
[LightGBM] [Info] Start training from score -1.641350
[LightGBM] [Info] Start training from score -1.225377
[LightGBM] [Info] Start training from score -2.747601
[LightGBM] [Info] Start training from score -3.340939
[LightGBM] [Info] Start training from score -2.471497
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf